# Modeling

In [1]:
import pandas as pd
import joblib

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_selector as selector, ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

In [2]:
churn_data = pd.read_csv('..//data//processed//churn_known.csv')
churn_test = pd.read_csv('..//data//processed//churn_holdout.csv')
churn_data.head()

,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_eve_minutes,total_eve_calls,total_night_minutes,total_night_calls,total_intl_minutes,total_intl_calls,number_customer_service_calls,churn
0,AR,110,area_code_408,False,False,0,55.3,102,164.7,124,200.7,108,10.2,5,1,False
1,WV,141,area_code_415,True,True,37,258.6,84,222.0,111,326.4,97,11.2,5,0,False
2,PA,122,area_code_415,True,False,0,230.9,132,243.2,99,182.4,57,11.0,2,0,True
3,ND,190,area_code_415,False,False,0,169.4,102,253.5,113,197.1,93,8.9,5,1,False
4,TN,68,area_code_415,False,False,0,178.7,61,252.3,84,255.7,76,8.4,4,1,False


In [3]:
data_train = churn_data
data_test = churn_test
target_col = 'churn'

X_train = data_train.drop(columns = target_col)
y_train = data_train[target_col]

X_test = data_test.drop(columns = target_col)
y_test = data_test[target_col]

num_cols_selector = selector(dtype_exclude = [object, bool])
cat_cols_selector = selector(dtype_include = [object, bool])

num_cols = num_cols_selector(X_train)
cat_cols = cat_cols_selector(X_train)

num_preprocess = StandardScaler()
cat_preprocess = OneHotEncoder(handle_unknown = 'ignore')

preprocessor = ColumnTransformer([('num_preprocess', num_preprocess, num_cols), ('cat_preprocess', cat_preprocess, cat_cols)])


## **I. Dummy**

In [4]:
def dummy(X_train, y_train, strategy = 'most_frequent'):

    pipeline = make_pipeline(preprocessor, DummyClassifier(strategy = strategy))

    pipeline.fit(X_train, y_train)
    
    return pipeline

In [5]:
dummy_model = dummy(X_train = X_train, y_train = y_train)
dummy_pred = dummy_model.predict(X_test)

In [6]:
print(classification_report(y_test, y_pred = dummy_pred))

              precision    recall  f1-score   support

       False       0.86      1.00      0.92       730
        True       0.00      0.00      0.00       120

    accuracy                           0.86       850
   macro avg       0.43      0.50      0.46       850
weighted avg       0.74      0.86      0.79       850



C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\PC\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\metrics\_classification.py:1509: Und

## **II. Logistic Regression**

In [7]:
random_state = 67
log_reg_model = LogisticRegression(max_iter=5000, random_state=random_state)
log_reg_params = {'logisticregression__C': loguniform(1e-4, 1e3), 
                  'logisticregression__penalty': ['l1', 'l2'], 
                  'logisticregression__solver': ['liblinear', 'saga'], 
                  'logisticregression__class_weight': [None, 'balanced']}

In [8]:
def opt_model(X_train, y_train, model, param_dist, random_state, n_iter=50, cv=5, scoring='f1'):

    pipeline = make_pipeline(preprocessor, model)
    
    search = RandomizedSearchCV(estimator=pipeline, param_distributions=param_dist, n_iter=n_iter, scoring=scoring, cv=cv, random_state=random_state, n_jobs=-1, verbose=1)
    search.fit(X_train, y_train)
    print(f'Best CV Score: {search.best_score_:.4f}')
    print(f'Best Parameters: {search.best_params_}')

    return search, search.best_score_

In [9]:
opt_log_model, opt_log_cv = opt_model(X_train = X_train, y_train = y_train, model = log_reg_model, param_dist = log_reg_params, random_state = random_state)
logist_pred = opt_log_model.predict(X_test)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV Score: 0.4886
Best Parameters: {'logisticregression__C': 0.03247586279643267, 'logisticregression__class_weight': 'balanced', 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'saga'}


In [10]:
print(classification_report(y_test, y_pred = logist_pred))

              precision    recall  f1-score   support

       False       0.95      0.76      0.84       730
        True       0.34      0.76      0.47       120

    accuracy                           0.76       850
   macro avg       0.65      0.76      0.66       850
weighted avg       0.86      0.76      0.79       850



## **III. Random Forest**

In [11]:
randomforest_model = RandomForestClassifier(random_state=random_state)
randomforest_params = {'randomforestclassifier__n_estimators': randint(100, 1000),
                       'randomforestclassifier__max_depth': randint(3, 30),
                       'randomforestclassifier__min_samples_split': randint(2, 20),
                       'randomforestclassifier__min_samples_leaf': randint(1, 20),
                       'randomforestclassifier__max_features': ['sqrt', 'log2', None],
                       'randomforestclassifier__bootstrap': [True, False],
                       'randomforestclassifier__criterion': ['gini', 'entropy', 'log_loss'],
                       'randomforestclassifier__class_weight': [None, 'balanced', 'balanced_subsample']}

In [12]:
opt_rf_model, opt_rf_cv = opt_model(X_train = X_train, y_train = y_train, model = randomforest_model, param_dist = randomforest_params, random_state = random_state)
rf_pred = opt_rf_model.predict(X_test)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV Score: 0.8258
Best Parameters: {'randomforestclassifier__bootstrap': True, 'randomforestclassifier__class_weight': 'balanced_subsample', 'randomforestclassifier__criterion': 'log_loss', 'randomforestclassifier__max_depth': 25, 'randomforestclassifier__max_features': None, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__min_samples_split': 8, 'randomforestclassifier__n_estimators': 424}


In [13]:
print(classification_report(y_test, y_pred = rf_pred))

              precision    recall  f1-score   support

       False       0.96      0.99      0.97       730
        True       0.90      0.78      0.83       120

    accuracy                           0.96       850
   macro avg       0.93      0.88      0.90       850
weighted avg       0.96      0.96      0.96       850



## **IV. XGBoost**

In [14]:
xgb_model = XGBClassifier(random_state=random_state, eval_metric='logloss')
xgb_params = {'xgbclassifier__n_estimators': randint(100, 1000),
                  'xgbclassifier__max_depth': randint(3, 10),
                  'xgbclassifier__learning_rate': uniform(0.01, 0.29),
                  'xgbclassifier__subsample': uniform(0.6, 0.4),
                  'xgbclassifier__colsample_bytree': uniform(0.6, 0.4),
                  'xgbclassifier__min_child_weight': randint(1, 10),
                  'xgbclassifier__gamma': uniform(0, 5),
                  'xgbclassifier__reg_alpha': uniform(0, 5),
                  'xgbclassifier__reg_lambda': uniform(0, 5),
                  'xgbclassifier__scale_pos_weight': uniform(4, 4)}

In [15]:
opt_xgb_model, opt_xgb_cv = opt_model(X_train = X_train, y_train = y_train, model = xgb_model, param_dist = xgb_params, random_state = random_state)
xgb_pred = opt_xgb_model.predict(X_test)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV Score: 0.8396
Best Parameters: {'xgbclassifier__colsample_bytree': 0.7346910282538062, 'xgbclassifier__gamma': 0.44727227172506323, 'xgbclassifier__learning_rate': 0.1472612369859015, 'xgbclassifier__max_depth': 8, 'xgbclassifier__min_child_weight': 3, 'xgbclassifier__n_estimators': 665, 'xgbclassifier__reg_alpha': 0.941703428607274, 'xgbclassifier__reg_lambda': 1.0170334945363968, 'xgbclassifier__scale_pos_weight': 4.622732881794799, 'xgbclassifier__subsample': 0.8052534599644118}


In [16]:
print(classification_report(y_test, y_pred = xgb_pred))

              precision    recall  f1-score   support

       False       0.97      0.99      0.98       730
        True       0.91      0.79      0.85       120

    accuracy                           0.96       850
   macro avg       0.94      0.89      0.91       850
weighted avg       0.96      0.96      0.96       850



## **V. CatBoost**

In [17]:
catboost_model = CatBoostClassifier(random_state=random_state, verbose=0)
catboost_params = {'catboostclassifier__iterations': randint(100, 1000),
                'catboostclassifier__depth': randint(3, 10),
                'catboostclassifier__learning_rate': uniform(0.01, 0.29),
                'catboostclassifier__l2_leaf_reg': uniform(1, 9),
                'catboostclassifier__bagging_temperature': uniform(0, 5),
                'catboostclassifier__random_strength': uniform(0, 5),
                'catboostclassifier__border_count': randint(32, 255),
                'catboostclassifier__auto_class_weights': ['Balanced', 'SqrtBalanced']}

In [18]:
opt_cat_model, opt_cat_cv = opt_model(X_train=X_train, y_train=y_train, model = catboost_model, param_dist = catboost_params, random_state = random_state)
cat_pred = opt_cat_model.predict(X_test)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV Score: 0.8555
Best Parameters: {'catboostclassifier__auto_class_weights': 'SqrtBalanced', 'catboostclassifier__bagging_temperature': 3.757120142089449, 'catboostclassifier__border_count': 229, 'catboostclassifier__depth': 4, 'catboostclassifier__iterations': 959, 'catboostclassifier__l2_leaf_reg': 1.539981366045561, 'catboostclassifier__learning_rate': 0.12202055650238558, 'catboostclassifier__random_strength': 1.0657492934217783}


In [19]:
print(classification_report(y_test, y_pred=cat_pred))

              precision    recall  f1-score   support

       False       0.96      0.99      0.98       730
        True       0.93      0.76      0.83       120

    accuracy                           0.96       850
   macro avg       0.95      0.87      0.91       850
weighted avg       0.96      0.96      0.96       850



## **VI. LightGBM**

In [20]:
lgbm_model = LGBMClassifier(random_state=random_state, verbose=-1)
lgbm_params = {'lgbmclassifier__n_estimators': randint(100, 1000),
                'lgbmclassifier__learning_rate': uniform(0.01, 0.29),
                'lgbmclassifier__max_depth': randint(3, 10),
                'lgbmclassifier__num_leaves': randint(20, 200),
                'lgbmclassifier__min_child_samples': randint(5, 100),
                'lgbmclassifier__subsample': uniform(0.6, 0.4),
                'lgbmclassifier__colsample_bytree': uniform(0.6, 0.4),
                'lgbmclassifier__reg_alpha': uniform(0, 5),
                'lgbmclassifier__reg_lambda': uniform(0, 5),
                'lgbmclassifier__scale_pos_weight': uniform(4, 4)}

In [21]:
opt_lgbm_model, opt_lgbm_cv = opt_model(X_train=X_train, y_train=y_train, model = lgbm_model, param_dist = lgbm_params, random_state = random_state)
lgbm_pred = opt_lgbm_model.predict(X_test)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best CV Score: 0.8457
Best Parameters: {'lgbmclassifier__colsample_bytree': 0.6340442445044466, 'lgbmclassifier__learning_rate': 0.18039646279213245, 'lgbmclassifier__max_depth': 9, 'lgbmclassifier__min_child_samples': 7, 'lgbmclassifier__n_estimators': 195, 'lgbmclassifier__num_leaves': 199, 'lgbmclassifier__reg_alpha': 0.5705130288464677, 'lgbmclassifier__reg_lambda': 3.334006016565918, 'lgbmclassifier__scale_pos_weight': 4.638952373650309, 'lgbmclassifier__subsample': 0.9056020040432478}


In [22]:
print(classification_report(y_test, y_pred=lgbm_pred))

              precision    recall  f1-score   support

       False       0.97      0.99      0.98       730
        True       0.94      0.78      0.85       120

    accuracy                           0.96       850
   macro avg       0.95      0.89      0.92       850
weighted avg       0.96      0.96      0.96       850



## **VII. Saving**

In [23]:
joblib.dump(opt_cat_model, "../models/catboost.joblib")
joblib.dump(opt_rf_model, "../models/random_forest.joblib")
joblib.dump(opt_xgb_model, "../models/xgboost.joblib")
joblib.dump(opt_lgbm_model, "../models/lightgbm.joblib")
joblib.dump(opt_log_model, "../models/logistic.joblib")

['../models/logistic.joblib']

In [24]:
opt_cv = {'LogisticRegression': opt_log_cv,
          'RandomForest': opt_rf_cv,
          'XGBoost': opt_xgb_cv,
          'CatBoost': opt_cat_cv,
          'LightGBM': opt_lgbm_cv}

opt_cv_df = pd.DataFrame.from_dict(opt_cv, orient='index', columns=['CV F1'])
opt_cv_df.to_csv('..//outputs//tables//opt_cv.csv')

In [25]:
actual_pred = {'Actual': y_test,
               'LogisticRegression': logist_pred,
               'RandomForest': rf_pred,
               'XGBoost': xgb_pred,
               'CatBoost': cat_pred,
               'LightGBM': lgbm_pred}

actual_pred_df = pd.DataFrame(actual_pred).astype(int)
actual_pred_df.to_csv('..//outputs//tables//actual_pred.csv', index = False)